# 📊 Day 3, Topics 9 & 10 — Binning Continuous Data & Creating Dummy Variables

Two feature engineering techniques that transform raw data into formats better suited for analysis and ML.


In [31]:
#Day 3, Topic 9: Binning Continuous Data
import pandas as pd
df = pd.read_csv('titanic.csv')

## Topic 9 — Binning Continuous Data

**Binning** (discretization) converts a continuous numeric column into discrete categories (bins).

### Why bin continuous data?
- Capture non-linear relationships (age groups behave differently, not linearly)
- Handle outliers gracefully (a fare of $500 falls in the 'Very High' bucket)
- Make features interpretable for reporting
- Required when an algorithm needs categorical input

### Two approaches

| Method | How bins are defined | Use when |
|--------|---------------------|---------|
| `pd.cut(col, bins=...)` | You specify the **bin edges** | Domain knowledge (e.g., 0–18 is 'Child') |
| `pd.qcut(col, q=N)` | Pandas finds edges so each bin has **equal count** | You want balanced classes |

### `pd.cut()` — Custom Edges
```python
pd.cut(df['Age'], bins=[0, 18, 35, 60, 100], labels=['Child','Young Adult','Adult','Senior'])
```
- Bins are `(left, right]` — left-exclusive, right-inclusive by default
- `include_lowest=True` makes the first bin `[left, right]` (includes the minimum)
- `labels=False` → returns integer bin codes (0, 1, 2…) instead of labels

### `pd.qcut()` — Equal-Frequency Bins
```python
pd.qcut(df['Fare'], q=4, labels=['Q1','Q2','Q3','Q4'])
```
- `q=4` → quartiles (each bin has ~25% of data)
- Edges are computed automatically from the data distribution
- Works poorly when many identical values exist (raises `ValueError`)


### `pd.cut()` — Fixed-Width Bins
`bins=[0, 18, 30, 50, 80]` defines 4 bins. Each value falls into exactly one bin.  
`labels` must have length = `len(bins) - 1`.  
The resulting column has dtype `category` (ordered by default).


In [32]:
#pd.cut() – Fixed‑Width or Custom Bins
bins = [0, 18, 30, 50, 80]
labels = ['Child', 'Young Adult', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
df[['Age', 'AgeGroup']].head(10)

,Age,AgeGroup
0,22.0,Young Adult
1,38.0,Adult
2,26.0,Young Adult
3,35.0,Adult
4,35.0,Adult
5,NaN,NaN
6,54.0,Senior
7,2.0,Child
8,27.0,Young Adult
9,14.0,Child


### `pd.qcut()` — Quantile-Based Bins
`q=4` creates quartiles — the 25th, 50th, and 75th percentiles become the edges.  
Each bin contains approximately the same number of passengers.  
`labels=['Q1','Q2','Q3','Q4']` — Q1 is the cheapest fares, Q4 the most expensive.


In [33]:
#pd.qcut() – Quantile‑Based Binning
df['FareQuantile'] = pd.qcut(df['Fare'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df['FareQuantile'].value_counts()

FareQuantile
Q2    224
Q1    223
Q3    222
Q4    222
Name: count, dtype: int64

### Handling Edge Cases — `include_lowest=True`
By default, `pd.cut` uses `(left, right]` intervals — the minimum value could fall outside if it exactly equals the left edge.  
`include_lowest=True` makes the first bin `[left, right]` — ensures no values are excluded.


In [34]:
#Handling Out‑of‑Bounds Values
pd.cut(df['Age'], bins=[0, 18, 50, 80], include_lowest=True).head(20)

0       (18.0, 50.0]
1       (18.0, 50.0]
2       (18.0, 50.0]
3       (18.0, 50.0]
4       (18.0, 50.0]
5                NaN
6       (50.0, 80.0]
7     (-0.001, 18.0]
8       (18.0, 50.0]
9     (-0.001, 18.0]
10    (-0.001, 18.0]
11      (50.0, 80.0]
12      (18.0, 50.0]
13      (18.0, 50.0]
14    (-0.001, 18.0]
15      (50.0, 80.0]
16    (-0.001, 18.0]
17               NaN
18      (18.0, 50.0]
19               NaN
Name: Age, dtype: category
Categories (3, interval[float64, right]): [(-0.001, 18.0] < (18.0, 50.0] < (50.0, 80.0]]

### Integer Bin Codes — `labels=False`
`pd.cut(df['Age'], bins=4, labels=False)` automatically computes 4 equal-width bins and returns **integer codes** (0, 1, 2, 3).  
Useful when you want numeric encodings without specifying edges or labels.


In [35]:
df['AgeBinCode'] = pd.cut(df['Age'], bins=4, labels=False)

---
### 📝 Practice — Binning


In [36]:
#Practice Activity: Binning
import pandas as pd 
df = pd.read_csv('titanic.csv')

#### Task 1 — Age Groups with Custom Bins
Bins `[0,12,18,35,60,100]` with labels `['Child','Teen','Young Adult','Adult','Senior']` — domain-meaningful age groups.


In [37]:
"""Task 1: Create an AgeGroup column using pd.cut() with bins [0, 12, 18, 35, 60, 100] and 
labels ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']. Display the value counts."""

bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
df['AgeGroup'].value_counts()

AgeGroup
Young Adult    358
Adult          195
Teen            70
Child           69
Senior          22
Name: count, dtype: int64

#### Task 2 — Fare Quantiles
`pd.qcut(df['Fare'], q=5, labels=[...])` → 5 equal-frequency fare bands.  
Note: Fare has many 0s and duplicates which may cause issues — use `duplicates='drop'` if needed.


In [38]:
#Task 2: Use pd.qcut() to divide Fare into 5 quantile bins. Add labels 
# ['Very Low', 'Low', 'Medium', 'High', 'Very High']. 
# Display the value counts.

pd.qcut(df['Fare'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']).value_counts()

Fare
Low          184
High         180
Very Low     179
Very High    176
Medium       172
Name: count, dtype: int64

#### Task 3 — Fare Categories with Custom Bins
`bins=[0,10,30,100,600]` captures meaningful price points: budget, moderate, expensive, luxury.


In [39]:
"""Task 3: Create a FareCategory column using pd.cut() with bins [0, 10, 30, 100, 600] and 
labels ['Cheap', 'Moderate', 'Expensive', 'Luxury']. How many passengers are in each category?"""

bins = [0, 10, 30, 100, 600]
labels = ['Cheap', 'Moderate', 'Expensive', 'Luxury']
df['FareCategory'] = pd.cut(df['Fare'], bins=bins, labels=labels)
df['FareCategory'].value_counts()

FareCategory
Cheap        321
Moderate     321
Expensive    181
Luxury        53
Name: count, dtype: int64

#### Task 4 — Auto Bins with Integer Codes
`pd.cut(df['Age'], bins=3, labels=False)` → 3 equal-width bins coded as 0, 1, 2.  
These integers represent bin indices, not actual values.


In [40]:
#Task 4: Use pd.cut() with bins=3 and labels=False on the Age column. What do the returned numbers represent?

pd.cut(df['Age'], bins=3, labels=False).head()

0    0.0
1    1.0
2    0.0
3    1.0
4    1.0
Name: Age, dtype: float64

#### Task 5 (Bonus) — FamilySize + Custom Binning
Create `FamilySize = SibSp + Parch + 1`, then bin into 'Solo'(1), 'Small'(2-4), 'Large'(5+).  
Demonstrates chaining feature engineering (new column) with binning.


In [41]:
"""Task 5 (Bonus): Create a new column FamilySize = SibSp + Parch + 1. Then use pd.cut() to 
categorize families as 'Solo' (1), 'Small' (2–3), 'Medium' (4–5), 'Large' (6+). Display the value counts."""

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
bins = [0, 1, 3, 5, 100]
labels = ['Solo', 'Small', 'Medium', 'Large']

df['FamilyCategory'] = pd.cut(df['FamilySize'], bins=bins, labels=labels)

df['FamilyCategory'].value_counts()

FamilyCategory
Solo      537
Small     263
Large      47
Medium     44
Name: count, dtype: int64

In [42]:
#Day 3, Topic 10: Creating Dummy Variables
import pandas as pd

df = pd.read_csv('titanic.csv')

## Topic 10 — Creating Dummy Variables (One-Hot Encoding)

Most ML algorithms require numeric input. Categorical columns like `Sex` ('male'/'female') or `Embarked` ('S'/'C'/'Q') must be converted to numbers.

### One-Hot Encoding
Creates one binary column per unique category value:
```
Sex: ['male', 'female']
→ Sex_male: [1, 0, 1, ...]
→ Sex_female: [0, 1, 0, ...]
```

### The Dummy Variable Trap
If you have k categories, you only need k-1 dummy columns.  
Reason: if all other dummies are 0, the dropped category is implied.  
Adding all k creates **perfect multicollinearity** — a problem for linear models.

`drop_first=True` automatically drops the first category.

### `pd.get_dummies()` Parameters
```python
pd.get_dummies(
    data,                # Series or DataFrame
    prefix='prefix',     # column name prefix (e.g., 'Sex_male', 'Sex_female')
    drop_first=True,     # drop first category (avoid dummy variable trap)
    columns=['col1'],    # which columns to encode (if data is a DataFrame)
    dtype=float          # dtype of output (default bool in newer Pandas)
)
```


### Basic `pd.get_dummies()` — Single Column
`pd.get_dummies(df['Sex'])` creates two columns: `female` and `male` (alphabetical order).  
Returns a new DataFrame — the original `df` is unchanged.


In [43]:
#Basic pd.get_dummies()
sex_dummies = pd.get_dummies(df['Sex'])
sex_dummies.head()

,female,male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True


### Adding a Prefix
`prefix='Sex'` → column names become `Sex_female` and `Sex_male`.  
Always use a prefix to avoid naming conflicts when concatenating with the original DataFrame.


In [44]:
#Adding a Prefix
sex_dummies = pd.get_dummies(df['Sex'], prefix='Sex')
sex_dummies.head()

,Sex_female,Sex_male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True


### Multiple Columns at Once
`pd.get_dummies(df[['Sex', 'Embarked']], prefix=['Sex', 'Emb'])` encodes both columns simultaneously.  
The result has one dummy per unique value across all specified columns.


In [45]:
#Creating Dummies for Multiple Columns
dumies = pd.get_dummies(df[['Sex', 'Embarked']], prefix=['Sex', 'Emb'])
dumies.head(10)

,Sex_female,Sex_male,Emb_C,Emb_Q,Emb_S
0,False,True,False,False,True
1,True,False,True,False,False
2,True,False,False,False,True
3,True,False,False,False,True
4,False,True,False,False,True
5,False,True,False,True,False
6,False,True,False,False,True
7,False,True,False,False,True
8,True,False,False,False,True
9,True,False,True,False,False


### `drop_first=True` — Avoid the Dummy Variable Trap
`drop_first=True` drops the alphabetically first category.  
For `Sex`: drops `Sex_female`, keeps `Sex_male` — `Sex_male=0` implies female.  
For linear regression and logistic regression, always use `drop_first=True`.


In [46]:
#Dropping the First Category (Avoid Dummy Variable Trap)
dumies = pd.get_dummies(df['Sex'], prefix='Sex', drop_first=True)
dumies.head(10)

,Sex_male
0,True
1,False
2,False
3,False
4,True
5,True
6,True
7,True
8,False
9,False


### Concatenating Dummies with the Original DataFrame
```python
sex_dummies = pd.get_dummies(df['Sex'], prefix='Sex')
df = pd.concat([df, sex_dummies], axis=1)
```
`axis=1` = concatenate as new **columns** (horizontal join).  
Then drop the original `'Sex'` column to avoid redundancy.


In [47]:
#Concatenating Dummies with Original DataFrame
df = pd.concat([df, sex_dummies], axis=1)


### `pd.get_dummies()` on the Whole DataFrame
`pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)` — encodes specified columns in one call.  
The original columns are replaced by their dummies automatically.


In [48]:
df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

---
### 📝 Practice — Dummy Variables


In [49]:
#Practice Activity: Dummy Variables

import pandas as pd
df = pd.read_csv('titanic.csv')

#### Task 1 — Dummies for Sex with Prefix
`pd.get_dummies(df['Sex'], prefix='Gender')` → `Gender_female` and `Gender_male`.


In [50]:
"""Task 1: Create dummy variables for the Sex column using pd.get_dummies(). 
Add the prefix 'Gender'. Display the first 5 rows."""

dummies = pd.get_dummies(df['Sex'], prefix='Gender')

dummies.head()

,Gender_female,Gender_male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True


#### Task 2 — Embarked Dummies with `drop_first`
`pd.get_dummies(df['Embarked'], prefix='Emb', drop_first=True)` → 2 columns instead of 3 (avoids trap).


In [51]:
"""Task 2: Create dummy variables for the Embarked column. 
Use drop_first=True to avoid the dummy variable trap. Display the resulting columns."""

dummies = pd.get_dummies(df['Embarked'], drop_first=True)

dummies.head()

,Q,S
0,False,True
1,False,False
2,False,True
3,False,True
4,False,True


#### Task 3 — Whole DataFrame Encoding
`pd.get_dummies(df, columns=['Sex','Pclass'], drop_first=True)` → adds dummy columns and drops originals.  
Count new columns with `df_encoded.shape[1] - df.shape[1]`.


In [52]:
"""Task 3: Use pd.get_dummies() on the entire DataFrame df, specifying columns=['Sex', 'Pclass'] 
and drop_first=True. How many new columns are created? (Hint: check .shape before and after.)"""

print(df.shape)

df_dummies = pd.get_dummies(df, columns=['Sex', 'Pclass'], drop_first=True)

df_dummies.shape

(891, 12)


(891, 13)

#### Task 4 — All Dummies then Check Sum
Create all Embarked dummies (no `drop_first`), then verify: for each row, the sum across all Embarked dummies = 1.  
`df_dummies.sum(axis=1).unique()` should return `[1]` — exactly one category per row.


In [53]:
"""Task 4: Create a new DataFrame df_dummies that contains only the dummy variables for Embarked (without dropping any) 
and then concatenate it back to the original df. Display the first 3 rows of the combined DataFrame."""

df_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked')

df_combined = pd.concat([df, df_dummies], axis=1)

df_combined.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,False,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,True,False,False
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,False,False,True


#### Task 5 (Bonus) — Deck from Cabin + Dummies
Extract deck letter (`df['Cabin'].str[0]`), fill NaN with 'Unknown', then encode with `pd.get_dummies()`.  
A full feature engineering pipeline: extract → clean → encode.


In [54]:
"""Task 5 (Bonus): The Cabin column contains values like 'C85', 'B28'. Extract the deck letter (first character) 
into a new column Deck. Then create dummy variables for Deck with drop_first=True. How many dummy columns are created?"""

df['Deck'] = df['Cabin'].str[0]
df_dummies = pd.get_dummies(df['Deck'], drop_first=True)

df_dummies.shape

(891, 7)